# ダーウィンが来た風ショート 生成ノートブック（Colab）

Colab の GPU 上で **ComfyUI（MiniMax H3）とパイプラインを同じマシンで動かします。**
リポジトリは Google ドライブに置くので、画像・生成物・設定はランタイムが切れても消えません。

実行順：上から順に。途中でランタイムが切れても、**生成済みのナレーション・クリップはスキップされる**ので、同じセルを再実行すれば続きから再開します。

## 1. 設定（最初に書き換える）

In [ ]:
EPISODE    = "01-keiba"
REPO_DIR   = "/content/drive/MyDrive/daniel"          # リポジトリの置き場所（ドライブ上）
BRANCH     = "claude/horse-racing-uncle-script-trojcd"
COMFY_DIR  = "/content/drive/MyDrive/ComfyUI"         # 水無瀬で使っている ComfyUI の場所に合わせる
COMFY_PORT = 8188

## 2. GPU 確認・ドライブ接続

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount("/content/drive")

## 3. リポジトリを取得／更新
リポジトリが非公開の場合は、Colab 左の🔑（シークレット）に `GITHUB_TOKEN`（repo 読み取り権限の Personal Access Token）を登録し、「ノートブックからのアクセス」をオンにしてください。

In [ ]:
import os, subprocess
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

token = secret("GITHUB_TOKEN")
url = f"https://{token}@github.com/bigdanclub/daniel.git" if token else "https://github.com/bigdanclub/daniel.git"
if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", "-b", BRANCH, url, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
os.chdir(f"{REPO_DIR}/darwin")
print(os.getcwd())

## 4. Gemini API キー（🔑 シークレットに `GEMINI_API_KEY` を登録しておく）

In [ ]:
os.environ["GEMINI_API_KEY"] = secret("GEMINI_API_KEY") or ""
print("GEMINI_API_KEY:", "OK" if os.environ["GEMINI_API_KEY"] else "未設定 ← 🔑 に登録してください")

## 5. ComfyUI を起動
`COMFY_DIR` に H3 が動く ComfyUI がある前提です。水無瀬のノートブックで毎回インストールしている場合は、**そのインストール用のセルをこのセルの直前に貼り付けて**、`COMFY_DIR` をその場所に合わせてください。

起動後に表示されるリンクから ComfyUI の画面を開けます（ワークフローの API 形式エクスポートに使います）。

In [ ]:
import sys, time, urllib.request
from google.colab import output

def comfy_up():
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{COMFY_PORT}/system_stats", timeout=3)
        return True
    except Exception:
        return False

if not comfy_up():
    log = open("/content/comfyui.log", "w")
    subprocess.Popen([sys.executable, "main.py", "--listen", "127.0.0.1", "--port", str(COMFY_PORT)],
                     cwd=COMFY_DIR, stdout=log, stderr=subprocess.STDOUT)
    for _ in range(300):
        if comfy_up():
            break
        time.sleep(2)
    else:
        raise SystemExit("ComfyUI が起動しません。/content/comfyui.log を確認してください。")
print("ComfyUI 起動済み")
output.serve_kernel_port_as_window(COMFY_PORT)

In [ ]:
# ComfyUI のログ末尾（エラー時の確認用）
!tail -n 30 /content/comfyui.log

## 6. 初回のみ：ワークフローとノード ID の設定
1. 上のリンクから ComfyUI を開き、H3 i2v ワークフローで **1回手動生成が通ること**を確認
2. **Workflow → Export (API)** でダウンロード → ドライブの `daniel/darwin/comfy/h3_i2v_api.json` にアップロード
3. 下のセルでノード一覧を表示し、`config.json` の `REPLACE_ME` を書き換える（ドライブ上で直接編集するか、下の2つ目のセルで書き換え）

In [ ]:
import json, shutil
wf_path = "comfy/h3_i2v_api.json"
if not os.path.exists("config.json"):
    shutil.copy("config.example.json", "config.json")
if os.path.exists(wf_path):
    for k, v in json.load(open(wf_path, encoding="utf-8")).items():
        print(f"{k:>5}  {v['class_type']:<32} {list(v['inputs'])}")
else:
    print("comfy/h3_i2v_api.json がまだありません（手順 6-2）")

In [ ]:
# ↓ 上の一覧を見てノードIDと入力名を書き換えて実行（初回のみ。以降は config.json に保存される）
cfg = json.load(open("config.json", encoding="utf-8"))
cfg["comfy"]["nodes"] = {
    "image":    {"node": "12", "input": "image"},
    "prompt":   {"node": "6",  "input": "text"},
    "negative": {"node": "7",  "input": "text"},     # 負プロンプトが無ければ "node": ""
    "duration": {"node": "30", "input": "length", "unit": "frames", "frame_offset": 1},
    "seed":     {"node": "3",  "input": "seed"},
}
cfg["comfy"]["fps"] = 24          # ワークフローの fps に合わせる
cfg["comfy"]["max_seconds"] = 15  # VRAM が足りなければ下げる（足りない尺はスローで埋める）
json.dump(cfg, open("config.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(json.dumps(cfg["comfy"], ensure_ascii=False, indent=2))

## 7. 画像の確認
Codex で作った画像を、ドライブの `daniel/darwin/episodes/01-keiba/images/` に
`01_basho.png` 〜 `06_ochi.png` の名前でアップロードしてください。
Codex 用のプロンプトは下のセルで書き出せます（`out/image_prompts.md`）。

In [ ]:
!python tools/prompts.py {EPISODE}
ep = json.load(open(f"episodes/{EPISODE}/episode.json", encoding="utf-8"))
for s in ep["sections"]:
    found = [e for e in (".png", ".jpg", ".jpeg", ".webp") if os.path.exists(f"episodes/{EPISODE}/images/{s['id']}{e}")]
    print(("✓ " if found else "✗ ") + s["id"] + (found[0] if found else "  ← 未アップロード"))

## 8. ① ナレーション（Gemini TTS）

In [ ]:
!python tools/tts_gemini.py {EPISODE}

In [ ]:
# 試聴（セクションIDを変えて確認）
from IPython.display import Audio
Audio(f"episodes/{EPISODE}/out/narration/01_basho.wav")

In [ ]:
# 1文だけ作り直す例（03 の 2文目）
# !python tools/tts_gemini.py {EPISODE} --redo 03_seitai:2

## 9. ② 映像（ComfyUI + MiniMax H3）
まず `--only 02_hakken` で1本だけ試し、問題なければ全クリップを生成します。

In [ ]:
!python tools/comfy_h3.py {EPISODE} --plan

In [ ]:
!python tools/comfy_h3.py {EPISODE} --only 02_hakken

In [ ]:
!python tools/comfy_h3.py {EPISODE}

In [ ]:
# 撮り直し例：シードを変えて1本だけ
# !python tools/comfy_h3.py {EPISODE} --redo 06_ochi --seed 12345

## 10. ③ 結合 → `out/final.mp4`（ドライブに保存されます）

In [ ]:
!python tools/assemble.py {EPISODE}

In [ ]:
# 手元にダウンロードする場合
from google.colab import files
files.download(f"episodes/{EPISODE}/out/final.mp4")